# 04 — LoRA adaptation for cash-flow stress

This is a classification adaptation, not a new pre-training run. The task is: given an account currently above a low-balance threshold fitted on training events, will it cross that threshold in the next 60 days? The target comes only from the future window; the model input stops at the cutoff.

The frozen probe was already competitive with the engineered baseline, which justifies a small adaptation experiment. The test set is not used to choose a rank or epoch.

## What changes during LoRA

All MLM backbone weights are frozen. The code wraps only History Encoder Q/K/V and MLP input/output projections with low-rank residuals. Event/Profile encoders, embeddings, LayerNorm, calendar projection, attention output projections, and MLM head remain frozen. A new linear head maps `account_embedding` to one logit.

Ranks 4, 8, and 16 are trained independently. `alpha=8`, dropout is zero, and LoRA-B begins at zero, so the adapter starts as the exact frozen backbone. Rank is chosen by validation average precision; the final selected adapter is then evaluated once on test.

In [2]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/FinancialBertForTransactions')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_ROOT = Path('/content/drive/MyDrive/FinancialBertForTransactions')
else:
    PERSIST_ROOT = PROJECT_ROOT
CHECKPOINT = PERSIST_ROOT / 'checkpoints_swoll' / 'pragma_lite_mlm' / 'best.pt'
REPORT_DIR = PERSIST_ROOT / 'reports'
TASK_TABLE = REPORT_DIR / 'cashflow_stress_task_table.parquet'
BASELINE_REPORT = REPORT_DIR / 'cashflow_stress_tabular_baseline.json'
FROZEN_REPORT = REPORT_DIR / 'cashflow_stress_frozen_probe.json'
ADAPTER_DIR = PERSIST_ROOT / 'adapters' / 'cashflow_stress'
assert all(path.exists() for path in (CHECKPOINT, TASK_TABLE, BASELINE_REPORT, FROZEN_REPORT)), 'Run notebook 03 first.'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
subprocess.run([
    sys.executable, 'scripts/run_lora_finetune.py',
    '--task', 'cashflow_stress',
    '--checkpoint', str(CHECKPOINT),
    '--task-table', str(TASK_TABLE),
    '--output-dir', str(ADAPTER_DIR),
    '--baseline-report', str(BASELINE_REPORT),
    '--frozen-probe-report', str(FROZEN_REPORT),
    '--ranks', '4,8,16', '--max-epochs', '50', '--patience', '8',
    '--device', 'cuda' if IN_COLAB else 'cpu',
], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_lora_finetune.py', '--task', 'cashflow_stress', '--checkpoint', '/content/drive/MyDrive/FinancialBertForTransactions/checkpoints_swoll/pragma_lite_mlm/best.pt', '--task-table', '/content/drive/MyDrive/FinancialBertForTransactions/reports/cashflow_stress_task_table.parquet', '--output-dir', '/content/drive/MyDrive/FinancialBertForTransactions/adapters/cashflow_stress', '--baseline-report', '/content/drive/MyDrive/FinancialBertForTransactions/reports/cashflow_stress_tabular_baseline.json', '--frozen-probe-report', '/content/drive/MyDrive/FinancialBertForTransactions/reports/cashflow_stress_frozen_probe.json', '--ranks', '4,8,16', '--max-epochs', '50', '--patience', '8', '--device', 'cuda'], returncode=0)

The generated JSON includes the three validation runs, selected rank, saved adapter-only checkpoint, tabular baseline, frozen probe, locked test metrics, and account-clustered intervals. Interpret a LoRA gain relative to both existing baselines; it is not enough merely to improve over the frozen model.